In [19]:
import torch 
import torch.optim as optim 
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torchvision import transforms, models, datasets 

In [5]:
train_40X = "Final-40X\\train" 
val_40X = "Final-40X\\val"

train_100X = "Final-100X\\train" 
val_100X = "Final-100X\\val"

train_200X = "Final-200X\\train"
val_200X = "Final-200X\\val" 

train_400X = "Final-400X\\train"
val_400X = "Final-400X\\val" 

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [7]:
model = models.densenet121(pretrained=True)

c:\Users\prath\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\prath\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
device = torch.device("cuda")


In [9]:
train_data = datasets.ImageFolder(train_40X, transform=transform)
val_data   = datasets.ImageFolder(val_40X, transform=transform)


In [10]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)


In [11]:
for param in model.features.parameters():
    param.requires_grad = False

In [12]:
num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Linear(num_features, 1),  # Binary classification
)

In [13]:
model = model.to(device)

In [18]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [20]:
def train_model(model, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        preds, true = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                outputs = torch.sigmoid(model(imgs))
                preds.extend((outputs.cpu().numpy() > 0.5).astype(int))
                true.extend(labels.numpy())

        acc = accuracy_score(true, preds)
        f1  = f1_score(true, preds)

        print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")



In [21]:
train_model(model, epochs=15)

Epoch 1/15 | Loss: 0.6697 | Acc: 0.7122 | F1: 0.7457
Epoch 2/15 | Loss: 0.5826 | Acc: 0.8098 | F1: 0.8251
Epoch 3/15 | Loss: 0.5209 | Acc: 0.8683 | F1: 0.8720
Epoch 4/15 | Loss: 0.4730 | Acc: 0.8878 | F1: 0.8930
Epoch 5/15 | Loss: 0.4345 | Acc: 0.8878 | F1: 0.8915
Epoch 6/15 | Loss: 0.4111 | Acc: 0.9024 | F1: 0.9078
Epoch 7/15 | Loss: 0.3877 | Acc: 0.9000 | F1: 0.9017
Epoch 8/15 | Loss: 0.3699 | Acc: 0.9024 | F1: 0.9065
Epoch 9/15 | Loss: 0.3506 | Acc: 0.9098 | F1: 0.9125
Epoch 10/15 | Loss: 0.3403 | Acc: 0.9098 | F1: 0.9142
Epoch 11/15 | Loss: 0.3279 | Acc: 0.9073 | F1: 0.9112
Epoch 12/15 | Loss: 0.3181 | Acc: 0.9024 | F1: 0.9065
Epoch 13/15 | Loss: 0.3073 | Acc: 0.9171 | F1: 0.9190
Epoch 14/15 | Loss: 0.2985 | Acc: 0.9220 | F1: 0.9242
Epoch 15/15 | Loss: 0.2923 | Acc: 0.9195 | F1: 0.9231


In [ ]:
model = models.densenet121(pretrained=False)  # create model architecture again
model.classifier = nn.Linear(model.classifier.in_features, 1)  

model.load_state_dict(torch.load("Densenet40X.pth", map_location=torch.device("gpu")))
model.eval()
